# Agent 365 Registry - Ingester (Fabric) — recommended default

> **Status: supported (app-only, unattended) — this is the recommended default path** for landing the
> Agent 365 registry into the Lakehouse. As of **PAX `purview-v1.11.11`** and the current
> Microsoft Graph docs, the *Agent 365 catalog* endpoint supports **application permissions** and
> **`/v1.0`**, so this notebook runs **headless on a schedule** with a service principal — no
> interactive sign-in. (It was previously a delegated-only, interactive PREVIEW.)

## What this does

Pulls the tenant's **Agent 365 agent catalogue** (declarative agents, plugins, etc. - including
their data-access **capabilities/permissions**: OneDrive/SharePoint read, Graph connector, code
interpreter, image generation, uploaded files) into the Lakehouse Delta table `dbo.agents_365`
(the table the dashboard reads), keyed on `Title ID` (the `T_`-prefixed titleId).

## Requirements & caveats (read before using)

| Item | Detail |
|---|---|
| **Auth** | **App-only / client credentials** (service principal). Runs unattended. No user sign-in. |
| **Permissions** | **Application** permissions `CopilotPackages.Read.All` **+** `Application.Read.All`, **admin-consented**. |
| **Agent 365 licence** | **Still required in the tenant.** This is a *SKU* check, separate from permissions - a missing licence returns **`403`** (`Customer must be licensed for Agent 365`). |
| **Endpoint version** | Uses **`/v1.0`** (GA). Automatically falls back to **`/beta`** if a tenant hasn't surfaced v1.0 yet (PAX itself still calls beta). |
| **Point-in-time only** | No history; deleted agents disappear; `Date created`/`Created by` need a separate Purview audit-log join (out of scope here). |

**Relationship to the export lander:** `Copilot_Agent365_Lander.ipynb` (admin-center **export CSV**)
is a **fallback** for tenants that can't grant the app-registration permissions this notebook needs,
or for one-off / evaluation runs. **Prefer this notebook** for scheduled production pipelines — you
get the live capability detail, no CSV upload step, and an automated refresh. The two notebooks are
**alternatives** — they write to the same `dbo.agents_365` table, so running both in the same pipeline
would just clobber each other.

## Setup

- An **app registration** (service principal) with the two **Application** permissions above,
  admin-consented, plus a **client secret** (store it in Key Vault / a Fabric secret, not in code)
  or a **certificate** / **managed identity**.
- Install MSAL only if you use the managed-identity path; the default client-secret path uses
  `requests` and needs nothing extra.

*Endpoint, 28-column schema and capability fields align with the PAX Agent 365 enrichment output so
the dashboard's `Agents 365` query reads them unchanged. Ref: Microsoft Graph
`copilot/admin/catalog/packages` (v1.0) and PAX `purview-v1.11.11`.*

## 1. Configuration & app-only sign-in

**App-only (client-credentials)** flow - runs unattended, so this notebook can be a scheduled Fabric
job. No device-code, no browser. The service principal's admin-consented **Application** permissions
(`CopilotPackages.Read.All` + `Application.Read.All`) are carried in the token via the `.default`
scope.

In [ ]:
# === CONFIG ===
TENANT_ID    = '<your-tenant-guid>'
CLIENT_ID    = '<app-reg-client-id>'      # Application perms: CopilotPackages.Read.All + Application.Read.All (admin-consented)
TARGET_TABLE = 'dbo.agents_365'   # canonical table the dashboard reads (matches the lander)
WRITE_MODE   = 'overwrite'

# Client secret - DO NOT hardcode. Pull from Key Vault / a Fabric-managed secret at runtime, e.g.:
#   CLIENT_SECRET = notebookutils.credentials.getSecret('https://<your-vault>.vault.azure.net/', 'Agent365AppSecret')
CLIENT_SECRET = '<from-key-vault>'

# App-only token via client credentials (mirrors Copilot_Audit_Log_Direct_Ingester.ipynb).
import requests

def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    data = {
        'client_id':     CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope':         'https://graph.microsoft.com/.default',
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=data)
    r.raise_for_status()
    return r.json()['access_token']

# Managed-identity alternative (no secret) - uncomment if the notebook runs under a UAMI/SAMI:
#   from azure.identity import DefaultAzureCredential
#   TOKEN = DefaultAzureCredential().get_token('https://graph.microsoft.com/.default').token

TOKEN = _get_graph_token()
print('app-only token acquired:', bool(TOKEN))

## 2. Call the v1.0 catalog endpoint

Uses **`/v1.0`** (GA, app-only) and **auto-falls back to `/beta`** if v1.0 isn't yet exposed in the
tenant. List -> page via `@odata.nextLink` -> fetch per-package detail (the `elementDetails`
capability fields only appear at the detail level).

- **`403`** = missing **Agent 365 licence** in the tenant (SKU check), *or* the app hasn't been
  admin-consented for `CopilotPackages.Read.All`.
- **`401`** = token/consent problem, not a code bug.

In [ ]:
import requests

API_VERSION = 'v1.0'   # GA + app-only; falls back to beta below if a tenant hasn't surfaced v1.0 yet
def _base(v):
    return f'https://graph.microsoft.com/{v}/copilot/admin/catalog/packages'
BASE = _base(API_VERSION)
H = {'Authorization': f'Bearer {TOKEN}'}

# Probe, with automatic v1.0 -> beta fallback (PAX itself still calls beta)
probe = requests.get(f'{BASE}?$top=1', headers=H)
if probe.status_code == 404 and API_VERSION == 'v1.0':
    API_VERSION = 'beta'
    BASE = _base(API_VERSION)
    probe = requests.get(f'{BASE}?$top=1', headers=H)
if probe.status_code == 403:
    raise PermissionError('403: Agent 365 catalog not accessible. Requires an Agent 365 LICENCE in '
                          'the tenant (SKU check) AND admin-consented Application permission '
                          'CopilotPackages.Read.All.')
if probe.status_code == 401:
    raise PermissionError('401: token/consent problem - check the app has admin consent for '
                          'CopilotPackages.Read.All + Application.Read.All (Application permissions).')
probe.raise_for_status()
print(f'Using {API_VERSION} endpoint.')

# Page the catalog list
packages, url = [], BASE
for _ in range(500):                       # safety cap on paging
    r = requests.get(url, headers=H); r.raise_for_status()
    data = r.json(); packages.extend(data.get('value', []))
    url = data.get('@odata.nextLink')
    if not url:
        break
print('packages in catalog:', len(packages))

## 3. Map to the 28-column schema → Delta

Column names match the PAX `ConvertTo-Agent365Row` output so the dashboard's `Agents 365` query can
read them. `Title ID` is the primary/merge key. `Date created` / `Created by` are left blank here
(they need a separate Purview audit-log join — out of scope for this preview).

In [ ]:
from pyspark.sql import functions as F

def _join(v):
    return ';'.join(v) if isinstance(v, list) else (v if v is not None else '')

rows = []
for pkg in packages:
    pid = pkg.get('id') or pkg.get('titleId') or pkg.get('packageId')
    d = requests.get(f'{BASE}/{pid}', headers=H)
    detail = d.json() if d.ok else pkg
    elem = detail.get('elementDetails') or {}
    tid = detail.get('id') or detail.get('titleId') or detail.get('packageId') or ''
    title_id = tid if str(tid).startswith('T_') else f'T_{tid}'
    rows.append({
        'Name': detail.get('displayName') or detail.get('name', ''),
        'Supported in': _join(detail.get('supportedHosts') or detail.get('supportedClients') or []),
        'Date created': '',
        'Developer Name': (detail.get('developer') or {}).get('name', ''),
        'Type': detail.get('agentType') or detail.get('type', ''),
        'Version': detail.get('version', ''),
        'Availability': _join(detail.get('availability') or detail.get('allowedUsersAndGroups') or ''),
        'Created by': '',
        'Description': detail.get('description', ''),
        'Created in': detail.get('source') or detail.get('origin') or detail.get('createdIn', ''),
        'Last updated': detail.get('lastModifiedDateTime') or detail.get('lastUpdatedDateTime', ''),
        'Custom actions': _join(elem.get('customActions', '')),
        'Title ID': title_id,
        'Sensitivity': detail.get('sensitivity', ''),
        'Can read OneDrive and Sharepoint items': elem.get('canReadOneDriveAndSharepointItems', ''),
        'OneDrive and Sharepoint items': _join(elem.get('oneDriveAndSharepointItems', '')),
        'Can read OneDrive files': elem.get('canReadOneDriveFiles', ''),
        'OneDrive files': _join(elem.get('oneDriveFiles', '')),
        'OneDrive sites': _join(elem.get('oneDriveSites', '')),
        'Can read Sharepoint sites and files': elem.get('canReadSharepointSitesAndFiles', ''),
        'Sharepoint files': _join(elem.get('sharepointFiles', '')),
        'Sharepoint sites': _join(elem.get('sharepointSites', '')),
        'Can extend to Graph connector': elem.get('canExtendToGraphConnector', ''),
        'Graph connector details': _join(elem.get('graphConnectorDetails', '')),
        'Can generate images using user prompt': elem.get('canGenerateImagesUsingUserPrompt', ''),
        'Can use code interpreter': elem.get('canUseCodeInterpreter', ''),
        'Contains uploaded files': elem.get('containsUploadedFiles', ''),
        'Uploaded files': _join(elem.get('uploadedFiles', '')),
    })

# everything as string for a clean, schema-stable Delta write
rows = [{k: ('' if v is None else str(v)) for k, v in r.items()} for r in rows]
df = spark.createDataFrame(rows) if rows else spark.createDataFrame([], 'Name string')

(df.write.mode(WRITE_MODE)
   .option('overwriteSchema', 'true')
   .option('delta.columnMapping.mode', 'name')      # spaced column names
   .option('delta.minReaderVersion', '2')
   .option('delta.minWriterVersion', '5')
   .format('delta').saveAsTable(TARGET_TABLE))

print(f'{TARGET_TABLE}: {df.count():,} agents written.')
print('NOTE: verify column names against the Agents 365 model query if the catalogue schema changes.')